# 第 10 章：MNIST 基础 CNN 配置对比

## Goal

在第 09 章全连接 MNIST 分类器的基础上，实现作业要求的 CNN：

```text
输入 (batch, 1, 28, 28)
→ [Conv2d → ReLU → MaxPool2d] × 3
→ Linear × 3
→ 输出 (batch, 10)
```

本 Notebook 会在相同训练设置下比较 CNN-S、CNN-M、CNN-L 三种通道与全连接层配置，并记录参数量、最佳准确率、最终准确率和训练时间。

![Exercise 10-1](./images/exercise_10_1.png)


## Setup

导入依赖并固定实验参数。

In [ ]:
from pathlib import Path
import gzip
import shutil
import time

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

BATCH_SIZE = 64
EPOCHS = 10
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")

## Data

复用第 09 章仓库内的 MNIST 压缩数据；首次运行时先解压 IDX 文件，再交给 `torchvision` 读取。

In [ ]:
def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "Chapter09_SoftmaxClassifier").exists():
            return candidate
    raise FileNotFoundError("请从 LiuErDaRenPyTorch 仓库内启动 Jupyter。")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
DATA_ROOT = (
    REPO_ROOT
    / "Chapter09_SoftmaxClassifier"
    / "MNISTHandwrittenDigitRecognition"
    / "data"
)


def ensure_raw_mnist(data_root: Path) -> None:
    raw_directory = data_root / "MNIST" / "raw"
    for archive in raw_directory.glob("*.gz"):
        target = raw_directory / archive.stem
        if not target.exists():
            with gzip.open(archive, "rb") as source, target.open("wb") as destination:
                shutil.copyfileobj(source, destination)


ensure_raw_mnist(DATA_ROOT)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = datasets.MNIST(
    root=DATA_ROOT,
    train=True,
    download=False,
    transform=transform,
)
test_dataset = datasets.MNIST(
    root=DATA_ROOT,
    train=False,
    download=False,
    transform=transform,
)


def make_train_loader(seed: int) -> DataLoader:
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        generator=generator,
    )


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

print(f"训练集: {len(train_dataset):,}")
print(f"测试集: {len(test_dataset):,}")

## Model

三个卷积块保持空间尺寸后再池化：`28 → 14 → 7 → 3`，因此 Flatten 后的特征数为 `channels[2] × 3 × 3`。

In [ ]:
class CNN(nn.Module):
    def __init__(
        self,
        channels=(16, 32, 64),
        hidden_dims=(128, 64),
    ):
        super().__init__()

        self.conv1 = nn.Conv2d(1, channels[0], kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2)

        self.conv2 = nn.Conv2d(channels[0], channels[1], kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2)

        self.conv3 = nn.Conv2d(channels[1], channels[2], kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=2)

        self.fc1 = nn.Linear(channels[2] * 3 * 3, hidden_dims[0])
        self.fc2 = nn.Linear(hidden_dims[0], hidden_dims[1])
        self.fc3 = nn.Linear(hidden_dims[1], 10)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))

        x = torch.flatten(x, start_dim=1)
        x = self.fc1(x)
        x = self.fc2(x)
        return self.fc3(x)


### Shape check

先验证模型输出尺寸，避免进入完整训练后才发现网络连接错误。

In [ ]:
shape_check_model = CNN()
shape_check_output = shape_check_model(torch.randn(4, 1, 28, 28))

print(shape_check_output.shape)
assert shape_check_output.shape == (4, 10)
assert CNN.forward is not nn.Module.forward

## Training and evaluation

训练与评估函数显式接收模型和优化器，避免依赖隐藏的全局状态。

In [ ]:
criterion = nn.CrossEntropyLoss()


def train_one_epoch(model, data_loader, optimizer):
    model.train()
    epoch_loss = 0.0

    for inputs, targets in data_loader:
        inputs = inputs.to(DEVICE)
        targets = targets.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(data_loader)


def evaluate(model):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            predictions = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (predictions == labels).sum().item()

    return 100.0 * correct / total

## Experiments

三种配置只改变通道数和隐藏层宽度；数据、随机种子、优化器和 epoch 数保持一致。首次调试可暂时把 `EPOCHS` 改为 2。

In [ ]:
configs = {
    "CNN-S": {"channels": (8, 16, 32), "hidden_dims": (64, 32)},
    "CNN-M": {"channels": (16, 32, 64), "hidden_dims": (128, 64)},
    "CNN-L": {"channels": (32, 64, 128), "hidden_dims": (256, 128)},
}

results = []
histories = {}

for name, config in configs.items():
    print(f"\n========== 开始训练 {name} ==========")

    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    train_loader = make_train_loader(SEED)
    model = CNN(**config).to(DEVICE)
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=0.01,
        momentum=0.5,
    )

    train_losses = []
    test_accuracies = []
    start_time = time.perf_counter()

    for epoch in range(EPOCHS):
        train_loss = train_one_epoch(model, train_loader, optimizer)
        test_accuracy = evaluate(model)

        train_losses.append(train_loss)
        test_accuracies.append(test_accuracy)

        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} | "
            f"loss={train_loss:.4f} | accuracy={test_accuracy:.2f}%"
        )

    elapsed = time.perf_counter() - start_time
    parameters = sum(parameter.numel() for parameter in model.parameters())

    histories[name] = {
        "loss": train_losses,
        "accuracy": test_accuracies,
    }
    results.append({
        "name": name,
        "params": parameters,
        "best_acc": max(test_accuracies),
        "final_acc": test_accuracies[-1],
        "time": elapsed,
    })

## Results

下表同时展示效果和计算成本。原第 09 章 MLP 的一次已记录结果为 97.73%，图中以虚线作为参考；CNN 指标必须以本 Notebook 实际运行输出为准。

In [ ]:
print(
    f"{'模型':<10}"
    f"{'参数量':>12}"
    f"{'最佳准确率':>14}"
    f"{'最终准确率':>14}"
    f"{'训练时间(s)':>14}"
)

for item in results:
    print(
        f"{item['name']:<10}"
        f"{item['params']:>12,}"
        f"{item['best_acc']:>13.2f}%"
        f"{item['final_acc']:>13.2f}%"
        f"{item['time']:>14.1f}"
    )

In [ ]:
epochs = range(1, EPOCHS + 1)
figure, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, history in histories.items():
    axes[0].plot(epochs, history["loss"], marker="o", label=name)
    axes[1].plot(epochs, history["accuracy"], marker="o", label=name)

axes[0].set(
    xlabel="Epoch",
    ylabel="Training Loss",
    title="Training Loss Comparison",
)
axes[1].axhline(
    y=97.73,
    color="gray",
    linestyle="--",
    label="Original MLP: 97.73%",
)
axes[1].set(
    xlabel="Epoch",
    ylabel="Test Accuracy (%)",
    title="Test Accuracy Comparison",
)

for axis in axes:
    axis.set_xticks(list(epochs))
    axis.grid(alpha=0.3)
    axis.legend()

figure.tight_layout()

output_path = REPO_ROOT / "Chapter10_BasicCNN" / "images" / "cnn_configuration_comparison.png"
output_path.parent.mkdir(parents=True, exist_ok=True)
figure.savefig(output_path, dpi=160, bbox_inches="tight")
plt.show()

print(f"对比图已保存至: {output_path}")

## Takeaways

运行完成后，从以下角度比较三种配置：

1. **准确率**：优先比较最佳测试准确率和最终测试准确率。
2. **稳定性**：观察准确率曲线是否持续提升、是否出现明显波动。
3. **成本**：结合参数量与训练时间判断扩大模型是否值得。
4. **推荐方式**：如果 CNN-M 与 CNN-L 准确率接近，而 CNN-M 参数更少、训练更快，则 CNN-M 是更均衡的选择。

> 注意：CNN 的实际准确率、运行时间与硬件和 PyTorch 版本有关，不应在运行前虚构实验结论。
